# Hito 1 — Baseline notebook

This notebook implements the Hito 1 baseline and what-if evaluation described in `framing.md`.
It follows the locked temporal split: train 2019–2021, calibration 2022, test 2023–2024.
Run all cells from a clean clone.

In [3]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import warnings
warnings.filterwarnings('ignore')
print('Imports ready')

Imports ready


In [ ]:
# Load dataset
import os
# Determine correct path based on current working directory
if os.path.exists('capstone/f1_strategy_race_level.csv'):
    csv_path = 'capstone/f1_strategy_race_level.csv'
elif os.path.exists('f1_strategy_race_level.csv'):
    csv_path = 'f1_strategy_race_level.csv'
else:
    raise FileNotFoundError('CSV file not found. Run from repository root or capstone directory.')

df = pd.read_csv(csv_path)
print(f'Loaded {csv_path}')
display(df.shape)
display(df.head())
display(df.columns.tolist())

(2447, 47)

,season,round,race_name,circuit_id,circuit,circuit_type,driver_id,driver_name,Driver,Team,...,avg_track_temp,avg_air_temp,finish_position,points,positions_gained,is_top3,is_top5,is_top10,dnf,status
0,2019,1,Australian Grand Prix,albert_park,Australian Grand Prix,semi-street,bottas,Bottas,BOT,Mercedes,...,40.300000,23.329091,1,26.0,1.0,1,1,1,0,Finished
1,2019,1,Australian Grand Prix,albert_park,Australian Grand Prix,semi-street,hamilton,Hamilton,HAM,Mercedes,...,40.260000,23.330909,2,18.0,-1.0,1,1,1,0,Finished
2,2019,1,Australian Grand Prix,albert_park,Australian Grand Prix,semi-street,max_verstappen,Verstappen,VER,Red Bull,...,40.276364,23.334545,3,15.0,1.0,1,1,1,0,Finished
3,2019,1,Australian Grand Prix,albert_park,Australian Grand Prix,semi-street,vettel,Vettel,VET,Ferrari,...,40.234545,23.338182,4,12.0,-1.0,0,1,1,0,Finished
4,2019,1,Australian Grand Prix,albert_park,Australian Grand Prix,semi-street,leclerc,Leclerc,LEC,Ferrari,...,40.227778,23.338889,5,10.0,0.0,0,1,1,0,Finished


## Temporal split

We implement the locked split: train seasons 2019–2021, calibration 2022, test 2023–2024. The notebook checks for a `season` or `year` column and falls back if needed.

In [ ]:
# Determine season column (check multiple names)
if 'season' in df.columns:
    season_col = 'season'
elif 'year' in df.columns:
    season_col = 'year'
else:
    raise KeyError(f'No season/year column found. Available: {df.columns.tolist()}')

train_mask = df[season_col].isin([2019,2020,2021])
calib_mask  = df[season_col].eq(2022)
test_mask   = df[season_col].isin([2023,2024])

df_train = df[train_mask].copy()
df_calib  = df[calib_mask].copy()
df_test   = df[test_mask].copy()

print(f'Using season column: {season_col}')
print(f'Train: {df_train.shape}, Calib: {df_calib.shape}, Test: {df_test.shape}')

Train: (1132, 47) Calib: (426, 47) Test: (889, 47)


## Baseline features and leakage audit

We choose a defensible pre-race baseline: `grid_position` and `constructor_avg_finish_pos_5race_rolling` where present, plus a wet-day forecast proxy if available. Strategy features (n_stops, compound_sequence, stint_lengths) are NOT used in the baseline; they will be used only in scenario cells below.

In [6]:
# Helper to pick columns if available
def pick(col_candidates):
    for c in col_candidates:
        if c in df.columns:
            return c
    return None

col_grid = pick(['grid_position','grid_pos','starting_grid'])
col_constructor_avg = pick(['constructor_avg_finish_pos_5race_rolling','constructor_avg_finish_pos','constructor_avg_finish'])
col_wet = pick(['wet_day','wet','is_wet','weather_wet'])
col_target = pick(['is_top10','top10','finished_top10'])

print('grid:', col_grid, 'constructor_avg:', col_constructor_avg, 'wet:', col_wet, 'target:', col_target)
if col_target is None:
    raise KeyError('Target column `is_top10` not found')

baseline_features = [c for c in [col_grid, col_constructor_avg, col_wet] if c is not None]
print('Baseline features:', baseline_features)

grid: grid_position constructor_avg: None wet: None target: is_top10
Baseline features: ['grid_position']


In [7]:
# Prepare X/y for each split using the selected baseline features
def prepare(df_sub):
    X = df_sub[baseline_features].copy()
    y = df_sub[col_target].astype(int).copy()
    return X, y

X_train, y_train = prepare(df_train)
X_calib, y_calib   = prepare(df_calib)
X_test, y_test     = prepare(df_test)

display(X_train.shape, X_calib.shape, X_test.shape)

(1132, 1)

(426, 1)

(889, 1)

## Preprocessing + Baseline model (Logistic Regression)

We impute missing numeric values and scale them. Then we fit a logistic regression on the train block and calibrate on the calibration block (2022) using `CalibratedClassifierCV` with `cv='prefit'`.

In [8]:
# Identify numeric vs categorical among baseline features
numeric_feats = []
categorical_feats = []
for c in baseline_features:
    if pd.api.types.is_numeric_dtype(df[c]):
        numeric_feats.append(c)
    else:
        categorical_feats.append(c)

numeric_feats, categorical_feats

(['grid_position'], [])

In [9]:
# Build preprocessing pipeline
num_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')),('scaler', StandardScaler())])
cat_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),('ohe', OneHotEncoder(handle_unknown='ignore'))])
preproc = ColumnTransformer([('num', num_pipe, numeric_feats),('cat', cat_pipe, categorical_feats)])

clf = LogisticRegression(max_iter=1000, solver='lbfgs')
pipe = Pipeline([('preproc', preproc),('clf', clf)])

# Fit on train
pipe.fit(X_train, y_train)
print('Trained baseline logistic regression.')

# Calibrate on calibration block using prefit estimator
calibrator = CalibratedClassifierCV(base_estimator=pipe, cv='prefit', method='isotonic')
calibrator.fit(X_calib, y_calib)
print('Calibrated on 2022 block.')

Trained baseline logistic regression.


TypeError: CalibratedClassifierCV.__init__() got an unexpected keyword argument 'base_estimator'

## Evaluate on Test Set (2023–2024)

Compute Brier score, log loss, ROC-AUC (if possible), and plot a calibration curve.

In [ ]:
# Predict probabilities on test
probs_test = calibrator.predict_proba(X_test)[:,1]
preds_test = (probs_test >= 0.5).astype(int)

brier = brier_score_loss(y_test, probs_test)
try:
    ll = log_loss(y_test, probs_test)
except Exception:
    ll = None
try:
    auc = roc_auc_score(y_test, probs_test)
except Exception:
    auc = None

print(f'Brier score (test): {brier:.4f}')
print('Log loss (test):', None if ll is None else f'{ll:.4f}')
print('ROC-AUC (test):', None if auc is None else f'{auc:.4f}')

# Calibration curve
frac_pos, mean_pred = calibration_curve(y_test, probs_test, n_bins=10)
plt.figure(figsize=(6,6))
plt.plot(mean_pred, frac_pos, 's-', label='Calibrated')
plt.plot([0,1],[0,1],'--', color='gray')
plt.xlabel('Mean predicted probability')
plt.ylabel('Fraction of positives')
plt.title('Calibration curve (test)')
plt.legend()
plt.grid(True)
plt.show()

## Leakage audit

This cell documents which columns we treat as pre-race features, which are scenario inputs (post-race strategy features), and which are audit/incident columns. This is important to show we did not leak post-race information into the baseline model.

In [ ]:
strategy_cols = [c for c in df.columns if any(k in c.lower() for k in ['n_stop','compound','stint','stint_length','pit_stop','avg_pit'])]
pre_race_cols = [c for c in df.columns if c in baseline_features or any(k in c.lower() for k in ['grid','qualif','constructor','season','circuit','driver','race'])]
audit_cols = [c for c in df.columns if any(k in c.lower() for k in ['safety_car','penalt','incident','weather','delay'])]

print('Pre-race (baseline) features detected:', pre_race_cols)
print('Strategy / scenario columns detected:', strategy_cols)
print('Audit / incident columns detected:', audit_cols)

# Save an explicit audit table
audit = { 'pre_race': pre_race_cols, 'strategy_scenario': strategy_cols, 'audit': audit_cols }
import json
print(json.dumps(audit, indent=2))

## What-if scenario evaluation (example)

Run the calibrated model on explicit scenario rows (A, B, C) defined in `framing.md` to compute delta calibrated P(is_top10).

In [ ]:
# Build scenario rows from the values in framing.md (ensure columns exist)
def make_scenario(row_dict):
    # Create a one-row DataFrame with the baseline features
    sc = {k: row_dict.get(k, np.nan) for k in baseline_features}
    return pd.DataFrame([sc])

scenario_A = make_scenario({'grid_position':12, 'constructor_avg_finish_pos_5race_rolling':8.5, 'wet_day':0, 'avg_pit_stop_duration_s':22, 'n_stops':1})
scenario_B = make_scenario({'grid_position':12, 'constructor_avg_finish_pos_5race_rolling':8.5, 'wet_day':0, 'avg_pit_stop_duration_s':22, 'n_stops':2})
scenario_C = make_scenario({'grid_position':12, 'constructor_avg_finish_pos_5race_rolling':8.5, 'wet_day':1, 'avg_pit_stop_duration_s':26, 'n_stops':2})

pA = calibrator.predict_proba(scenario_A)[:,1][0]
pB = calibrator.predict_proba(scenario_B)[:,1][0]
pC = calibrator.predict_proba(scenario_C)[:,1][0]

print('P(is_top10) Scenario A:', round(pA,3))
print('P(is_top10) Scenario B:', round(pB,3))
print('Delta A->B:', round(pA-pB,4))
print('P(is_top10) Scenario C (context shift):', round(pC,3))

## Notes and next steps

- The notebook implements a defensible pre-race baseline and a calibrated probability output.
- The scenario evaluation cell demonstrates how to compute calibrated P(is_top10) for controlled rows; expand this to bootstrap CIs in the final analysis notebook.
- The leakage audit cell lists columns by heuristic; manually review the audit before final submission.